# ☁️ ConPL: Consistent Prototype Learning (ACL 2023) — Google Colab
### Few-Shot Continual Relation Extraction on FewRel 5-shot (8 tasks × 10 relations, 6 sequences)

Notebook này được cấu hình để chạy trên **Google Colab với GPU NVIDIA**:
- **Codebase**: Tự tìm repository hiện có hoặc clone nhánh `feat/reproduce-conpl-baseline` vào runtime Colab.
- **Dataset chuẩn**: FewRel 10-way 5-shot (8 tasks, mỗi task 10 relations, 5-shot toàn bộ 8 tasks).
- **Backbone**: `bert-base-uncased` kết hợp Prompt-based Tuning (`e1 mask e2 . `).
- **Chế độ chạy linh hoạt**: Hỗ trợ chạy nhanh `smoke_test` (1 sequence, 2 tasks) hoặc tái lập trọn vẹn `full_benchmark` (6 sequences, 8 tasks theo Table 1 bài báo).


## 1. Kiểm tra Thiết bị Tính toán (Local NVIDIA T4 GPU Preflight)


In [ ]:
import sys
import torch

print('=' * 60)
print('KIỂM TRA THIẾT BỊ TÍNH TOÁN (LOCAL T4 GPU PREFLIGHT)')
print('=' * 60)
print(f'✓ Python Version: {sys.version.split()[0]} ({sys.executable})')

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✓ CUDA Khả dụng: True')
    print(f'✓ GPU Thiết bị:  {device_name}')
    print(f'✓ Dung lượng VRAM: {vram_gb:.2f} GB')
    print(f'✓ Phiên bản CUDA PyTorch: {torch.version.cuda}')
    !nvidia-smi
else:
    print('⚠️ CẢNH BÁO: CUDA không khả dụng trên kernel hiện tại. Vui lòng kiểm tra lại driver NVIDIA và cấu hình GPU của kernel!')


## 2. Thiết lập Không gian Làm việc & Định vị Thư mục ConPL


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
REPO_NAME = 'research-code-lab'
REPO_BRANCH = 'feat/reproduce-conpl-baseline'

current_dir = Path.cwd().resolve()
print(f'📍 Thư mục hiện tại của Kernel: {current_dir}')

# 1. Danh sách các đường dẫn ứng viên tiềm năng
candidates = [
    current_dir / 'paper-reproductions' / 'ConPL',
    current_dir.parent / 'paper-reproductions' / 'ConPL',
    current_dir / REPO_NAME / 'paper-reproductions' / 'ConPL',
    current_dir.parent / REPO_NAME / 'paper-reproductions' / 'ConPL',
    Path.home() / REPO_NAME / 'paper-reproductions' / 'ConPL',
    Path.home() / 'code' / REPO_NAME / 'paper-reproductions' / 'ConPL',
    Path('/content') / REPO_NAME / 'paper-reproductions' / 'ConPL',
    Path('/content') / 'paper-reproductions' / 'ConPL',
]

conpl_dir = None
for p in candidates:
    if (p / 'fewrel_5shot.py').exists() and (p / 'dataprocess.py').exists():
        conpl_dir = p.resolve()
        break

# 2. Nếu chưa thấy trong các đường dẫn quen thuộc, tìm kiếm trong cây thư mục hiện tại & Home
if conpl_dir is None:
    print('🔍 Đang tìm kiếm mã nguồn ConPL trên máy local...')
    for root in [current_dir, Path.home()]:
        try:
            matches = list(root.glob('**/paper-reproductions/ConPL/fewrel_5shot.py'))
            if matches:
                conpl_dir = matches[0].parent.resolve()
                break
        except Exception:
            pass

# 3. Nếu vẫn không thấy, tự động clone repository về thư mục làm việc của Kernel
if conpl_dir is None:
    print(f'⚠️ Không tìm thấy repository trên máy. Đang tự động clone từ {REPO_URL}...')
    target_repo = current_dir / REPO_NAME
    !git clone {REPO_URL} "{target_repo}"
    candidate_new = target_repo / 'paper-reproductions' / 'ConPL'
    if candidate_new.exists():
        conpl_dir = candidate_new.resolve()

if conpl_dir is None or not conpl_dir.exists():
    raise FileNotFoundError(
        'Vẫn không tìm thấy thư mục ConPL! Vui lòng chỉ định thủ công biến conpl_dir trỏ đến thư mục chứa fewrel_5shot.py.'
    )

print(f'✓ Thư mục làm việc ConPL: {conpl_dir}')
%cd {conpl_dir}

# Cập nhật repo đang có trên GPU từ nhánh chứa các bản vá ConPL.
repo_root = next((parent for parent in conpl_dir.parents if (parent / '.git').exists()), None)
if repo_root is None:
    raise FileNotFoundError(f'Không tìm thấy Git repository cha của {conpl_dir}')

def run_git(*args):
    return subprocess.run(
        ['git', '-C', str(repo_root), *args],
        check=True,
        text=True,
        capture_output=True,
    )

print(f'🔄 Đang cập nhật code từ origin/{REPO_BRANCH}...')
run_git('fetch', '--quiet', 'origin', REPO_BRANCH)
local_branch = subprocess.run(
    ['git', '-C', str(repo_root), 'show-ref', '--verify', '--quiet', f'refs/heads/{REPO_BRANCH}'],
    check=False,
)
if local_branch.returncode == 0:
    run_git('checkout', REPO_BRANCH)
else:
    run_git('checkout', '--track', '-b', REPO_BRANCH, f'origin/{REPO_BRANCH}')
run_git('pull', '--ff-only', 'origin', REPO_BRANCH)
print(f'✓ Đã cập nhật code lên nhánh {REPO_BRANCH}')

# Kiểm tra các tệp dữ liệu cốt lõi
required_files = [
    'data/fewrel/CFRLdata_10_100_10_5/rel_cluster_label_0.npy',
    'data/fewrel/CFRLdata_10_100_10_5/train_0.txt',
    'data/fewrel/CFRLdata_10_100_10_5/valid_0.txt',
    'data/fewrel/CFRLdata_10_100_10_5/test_0.txt',
    'config/config_fewrel_5and10.json'
]
all_exist = all(Path(f).exists() for f in required_files)
if all_exist:
    print('✓ Toàn bộ dữ liệu FewRel 5-shot và config đã sẵn sàng.')
else:
    print('⚠️ Một số file dữ liệu chưa đủ, vui lòng kiểm tra lại thư mục data/fewrel/')

## 3. Kiểm tra Phụ thuộc (Dependencies Verification)


In [ ]:
# Kiểm tra và cài đặt chỉ những thư viện còn thiếu (không đụng tới numpy đã có sẵn của kernel)
import importlib
import numpy as np

# Vá lỗi tương thích ragged array cho các phiên bản NumPy hiện đại (>= 1.24)
# Keep the true NumPy function across cell re-runs; otherwise wrappers recurse.
if not hasattr(np, "_conpl_original_asarray"):
    np._conpl_original_asarray = np.asarray
_orig_asarray = np._conpl_original_asarray
def _safe_asarray(a, *args, **kwargs):
    try:
        return _orig_asarray(a, *args, **kwargs)
    except ValueError:
        return _orig_asarray(a, dtype=object, *args, **kwargs)
np.asarray = _safe_asarray

required_packages = {
    'torch': 'torch',
    'numpy': 'numpy',
    'sklearn': 'scikit-learn',
    'wordninja': 'wordninja',
    'six': 'six',
    'tqdm': 'tqdm'
}

missing = []
for module_name, pip_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    pkg_str = ' '.join(missing)
    print(f'⚠️ Thiếu thư viện: {missing}. Đang tiến hành cài đặt...')
    !pip install -q {pkg_str}
    print(f'✓ Đã cài đặt thành công: {missing}')
else:
    print('✓ Toàn bộ thư viện phụ thuộc đã có sẵn trên kernel!')


## 4. Xác thực Dữ liệu FewRel & Sampler Integrity (Preflight Check)


In [ ]:
import sys
import json
import torch
import numpy as np
import inspect
from types import ModuleType

# 1. Vá lỗi tương thích ragged array cho NumPy hiện đại (>= 1.24)
# Keep the true NumPy function across cell re-runs; otherwise wrappers recurse.
if not hasattr(np, "_conpl_original_asarray"):
    np._conpl_original_asarray = np.asarray
_orig_asarray = np._conpl_original_asarray
def _safe_asarray(a, *args, **kwargs):
    try:
        return _orig_asarray(a, *args, **kwargs)
    except ValueError:
        return _orig_asarray(a, dtype=object, *args, **kwargs)
np.asarray = _safe_asarray

# 2. Vá lỗi tương thích cho huggingface_hub hiện đại (>= 1.0)
try:
    import huggingface_hub
    import huggingface_hub.file_download
    from tqdm.auto import tqdm

    # Vá HfFolder
    if not hasattr(huggingface_hub, 'HfFolder'):
        class HfFolder:
            @staticmethod
            def get_token():
                return getattr(huggingface_hub, 'get_token', lambda: None)()
            @staticmethod
            def save_token(token):
                pass
        huggingface_hub.HfFolder = HfFolder

    # Vá tqdm trong file_download để không lỗi AttributeError
    if not hasattr(huggingface_hub.file_download, 'tqdm'):
        huggingface_hub.file_download.tqdm = tqdm

    # Vá tham số use_auth_token thành token cho hf_hub_download
    if 'token' in inspect.signature(huggingface_hub.hf_hub_download).parameters:
        _orig_download = huggingface_hub.hf_hub_download
        def _compat_download(*args, **kwargs):
            if 'use_auth_token' in kwargs:
                tok = kwargs.pop('use_auth_token')
                if tok not in (None, False, True):
                    kwargs['token'] = tok
            return _orig_download(*args, **kwargs)
        huggingface_hub.hf_hub_download = _compat_download
except Exception:
    pass

# 3. Bỏ qua runtime dependency version assertion cũ của transformers trên Python 3.13
dummy_check = ModuleType('transformers.dependency_versions_check')
dummy_check.dep_version_check = lambda *a, **k: None
sys.modules['transformers.dependency_versions_check'] = dummy_check

# 4. Đảm bảo import ưu tiên từ thư mục ConPL hiện tại
if '.' not in sys.path:
    sys.path.insert(0, '.')

from encode import BERTSentenceEncoderPrompt
from dataprocess import data_sampler_bert_prompt_deal_first_task

# Patch the exact function reference used by legacy Transformers cached_file().
import transformers.utils.hub as _transformers_hub
_hf_download_impl = _transformers_hub.hf_hub_download
def _hub_download_compat(*args, **kwargs):
    if 'use_auth_token' in kwargs:
        kwargs['token'] = kwargs.pop('use_auth_token')
    return _hf_download_impl(*args, **kwargs)
_transformers_hub.hf_hub_download = _hub_download_compat

with open('config/config_fewrel_5and10.json', 'r') as f:
    cfg = json.load(f)

cfg['rel_cluster_label'] = 'data/fewrel/CFRLdata_10_100_10_5/rel_cluster_label_0.npy'
cfg['training_file'] = 'data/fewrel/CFRLdata_10_100_10_5/train_0.txt'
cfg['valid_file'] = 'data/fewrel/CFRLdata_10_100_10_5/valid_0.txt'
cfg['test_file'] = 'data/fewrel/CFRLdata_10_100_10_5/test_0.txt'
cfg['first_task_k-way'] = 10
cfg['k-shot'] = 5
cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Đang nạp BERT Prompt Sentence Encoder và khởi tạo Sampler...')
encoder = BERTSentenceEncoderPrompt(cfg)
sampler = data_sampler_bert_prompt_deal_first_task(cfg, encoder.tokenizer, 'e1 mask e2 . ')
sampler.set_seed(100)

print('=' * 60)
print('XÁC THỰC DỮ LIỆU & ENCODER THÀNH CÔNG')
print('=' * 60)
print(f'✓ Thiết bị sử dụng: {cfg["device"]}')
print(f'✓ Số lượng quan hệ (Relations): {len(sampler.id2rel) - 1}')
print(f'✓ Kích thước từ vựng (Vocab size): {len(encoder.tokenizer)}')
print('✓ Data Sampler khởi tạo hoàn toàn chính xác!')


## 5. Cấu hình & Thực thi Huấn luyện ConPL trên GPU T4


In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/ConPL_results")

# ==============================================================================
# CẤU HÌNH CHẾ ĐỘ CHẠY:
# - MODE = 'smoke_test': Chạy thử 1 sequence, 2 tasks (~2-3 phút) để test pipeline.
# - MODE = 'full_benchmark': Chạy đầy đủ 6 sequences, 8 tasks theo Table 1 bài báo ConPL.
# ==============================================================================
MODE = 'full_benchmark'  # Đổi thành 'smoke_test' nếu muốn kiểm thử nhanh trước

if MODE == 'smoke_test':
    os.environ['CONPL_SEQUENCES'] = '1'
    os.environ['CONPL_TASKS'] = '2'
    out_dir = str(DRIVE_OUTPUT_ROOT / 'colab_smoke')
else:
    os.environ['CONPL_SEQUENCES'] = '6'
    os.environ['CONPL_TASKS'] = '8'
    out_dir = str(DRIVE_OUTPUT_ROOT / 'colab_run_6seq')

os.environ['CONPL_DEVICE'] = 'cuda'
os.environ['CONPL_OUTPUT_DIR'] = out_dir
Path(out_dir).mkdir(parents=True, exist_ok=True)

log_path = Path(out_dir) / 'run.log'
print(f'🚀 Bắt đầu thực thi ConPL ({MODE}) trên GPU Colab...')
print(f'📁 Kết quả, metrics và checkpoints sẽ được lưu trên Drive tại: {out_dir}')
print(f'📝 File log chi tiết: {log_path}')
print('=' * 60)

!bash -o pipefail -c 'python fewrel_5shot.py 2>&1 | tee "{log_path}"'


## 6. Phân tích Kết quả & Đối chiếu Paper Table 1 (ACL 2023)


In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

paper_reported = {
    'T1': 95.72, 'T2': 93.53, 'T3': 91.31, 'T4': 89.95,
    'T5': 88.93, 'T6': 88.39, 'T7': 87.43, 'T8': 85.77
}

output_dir = Path(os.environ.get('CONPL_OUTPUT_DIR', '/content/drive/MyDrive/ConPL_results/colab_run_6seq'))
metrics_files = sorted(output_dir.glob('seed_*/task_metrics.jsonl'))

print('=' * 60)
print('TỔNG HỢP KẾT QUẢ THỰC NGHIỆM CONPL')
print('=' * 60)

if metrics_files:
    seq_accs = {}
    for mf in metrics_files:
        seed_name = mf.parent.name
        with open(mf, 'r', encoding='utf-8') as f:
            records = [json.loads(line) for line in f]
        seq_accs[seed_name] = {f"T{r['task']}": r.get('accuracy_whole', r.get('whole_acc')) * 100 for r in records}

    df = pd.DataFrame(seq_accs).T
    df.loc['Colab Mean'] = df.mean()
    df.loc['Paper Table 1'] = pd.Series(paper_reported)
    if 'Colab Mean' in df.index and 'Paper Table 1' in df.index:
        df.loc['Diff (Local - Paper)'] = df.loc['Colab Mean'] - df.loc['Paper Table 1']

    display(df.round(2))

    # Vẽ biểu đồ đường cong độ chính xác tích lũy
    plt.figure(figsize=(9, 5))
    tasks = list(paper_reported.keys())
    plt.plot(tasks, [paper_reported[t] for t in tasks], 'o--', color='#d9534f', label='Paper Reported (Table 1 Mean)')

    local_means = [df.loc['Colab Mean'].get(t, None) for t in tasks]
    valid_tasks = [t for t, v in zip(tasks, local_means) if pd.notnull(v)]
    valid_vals = [v for v in local_means if pd.notnull(v)]
    if valid_vals:
        plt.plot(valid_tasks, valid_vals, 's-', color='#2e6da4', linewidth=2, label='Colab Mean')

    plt.xlabel('Continual Tasks', fontsize=12)
    plt.ylabel('Cumulative Whole Accuracy (%)', fontsize=12)
    plt.title('ConPL FewRel 5-shot: Colab vs. Paper Table 1', fontsize=13, fontweight='bold')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.ylim(80, 100)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plot_file = output_dir / 'accuracy_comparison.png'
    plt.savefig(plot_file, dpi=200)
    plt.show()
    print(f'✓ Đã lưu biểu đồ đối chiếu tại: {plot_file}')
else:
    print('Chưa có file task_metrics.jsonl được tạo. Giá trị bài báo công bố:')
    for k, v in paper_reported.items():
        print(f'{k}: Paper={v:.2f}%')
